# M24 — equal-memory Packed Exact and FD-Ridge controls

This train-only notebook adds the two reviewer-facing controls preregistered in the M24 protocol. It never creates or reads `test.pt`. Use a GPU runtime; A100 is preferred because exact Frequent Directions performs repeated SVDs. Each stream is a separate resumable cell and downloads its JSON immediately after completion.

Upload the exact M23 artifact when requested. On a resumed runtime, upload any previously downloaded `stream_2025_results.json`, `stream_2026_results.json`, or `stream_2027_results.json` in the same upload dialog.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='ece2a1c5a7c15bf0888e622769fdfe4ef282865b'
WORK_DIR='/content/SOHO-CL'
RUN_ROOT='/content/srq_m24'
FEATURE_CACHE_DIR=RUN_ROOT+'/cifar_train_features'
OUTPUT_DIR=RUN_ROOT+'/output'
CONFIG='configs/srq_generalization_m24_equal_memory_controls_train_only.json'
RUNNER='tools/srq_generalization_m24.py'
M23_NAME='srq_generalization_m23_equal_budget_multistream_train_only.zip'
M23_SHA='80c8940a1d7ff11fd71b14c127802ca1a80bc76aec608cbd93d42fc8a8595b2a'
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
FINAL_EXPORT='/content/srq_generalization_m24_equal_memory_controls_train_only.zip'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh pinned checkout, dependencies, GPU check, and source locks.
import hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
def sha_raw(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
os.chdir('/content')
if Path(WORK_DIR).exists(): shutil.rmtree(WORK_DIR)
subprocess.run(['git','clone','--no-checkout','--quiet',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach','--quiet',REPO_COMMIT],cwd=WORK_DIR,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK_DIR,text=True).strip()==REPO_COMMIT
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Runtime > Change runtime type > choose a GPU, then restart from cell 1.'
print('GPU:',torch.cuda.get_device_name(0))
EXPECTED_SOURCE={
 'configs/srq_generalization_m24_equal_memory_controls_train_only.json':'09061bb9b73f2eb34557ed6f0245b8724068d92b844659da742baa6c76c8559d',
 'tools/srq_generalization_m24.py':'319ccd3e7ebc4c9c918df786d5e00a076d1807495e12d23f1e94f20ea743a5c3',
 'methods/analytic_ridge/equal_memory_controls.py':'98e9e0a6d3c0354fbe6b84a572daff4391be7c8337a133d8fb0f6a33f8139717',
 'methods/analytic_ridge/__init__.py':'204a4ca09cf7a3d400e8b08e44bf6b0ca9b2e1ddaf61fded9e678c33f7bd476e',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c'
}
for path,expected in EXPECTED_SOURCE.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Pinned checkout must start clean.'
print('M24 PINNED SOURCES: PASS')

In [ ]:
# Focused correctness gates; no CIFAR feature or test data is read.
command=[sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_equal_memory_controls.py','tests/test_srq_generalization_m24.py','tests/test_analytic_ridge_backend.py','tests/test_ranpac_analytic_frontend.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M24 preflight tests failed; preserve the complete traceback.'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Tests changed the pinned checkout.'
print('M24 PREFLIGHT TESTS: PASS')

In [ ]:
# Upload the exact M23 source artifact and optionally prior M24 stream JSON handoffs.
from google.colab import files
os.chdir('/content')
uploaded=files.upload()
assert M23_NAME in uploaded,'Upload the exact M23 ZIP with its original filename.'
M23_PATH=Path('/content')/M23_NAME
assert M23_PATH.is_file() and sha_raw(M23_PATH)==M23_SHA,(M23_PATH,sha_raw(M23_PATH) if M23_PATH.is_file() else None,M23_SHA)
stream_dir=Path(OUTPUT_DIR)/'m24_results'
stream_dir.mkdir(parents=True,exist_ok=True)
for seed in (2025,2026,2027):
    name=f'stream_{seed}_results.json'
    candidate=Path('/content')/name
    if candidate.is_file(): shutil.move(str(candidate),str(stream_dir/name))
os.chdir(WORK_DIR)
assert not (Path(WORK_DIR)/M23_NAME).exists(),'Keep uploaded artifacts outside the pinned checkout.'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Uploads dirtied the pinned checkout.'
print('M23 VERIFIED; RESTORED STREAMS:',sorted(p.name for p in stream_dir.glob('stream_*_results.json')))

In [ ]:
# Download locked ViT-B/16 and materialize CIFAR-100 train features only.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE
assert sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m24','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN-ONLY CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Helper: run one stream, verify structural PASS, and download its resume handoff.
def run_and_download_stream(stream_id,seed):
    command=[sys.executable,'-u',RUNNER,'run-stream','--stream-id',stream_id,'--config',CONFIG,'--source-m23-artifact',str(M23_PATH),'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
    print('M24 START/RESUME:',stream_id,flush=True)
    completed=subprocess.run(command)
    result_path=Path(OUTPUT_DIR)/'m24_results'/f'stream_{seed}_results.json'
    assert result_path.is_file(),f'{stream_id} failed before writing a handoff; preserve the complete runner output.'
    result=json.loads(result_path.read_text())
    print('STATUS:',result['status']); print('GATES:',json.dumps(result['gates'],indent=2))
    assert completed.returncode==0 and result['status']=='PASS_M24_EQUAL_MEMORY_CONTROLS_STREAM_TRAIN_ONLY',f'{stream_id} failed structural gates; do not relax gates.'
    files.download(str(result_path))
    return result

## Stream s2025
Run this cell and keep the downloaded JSON. FD-Ridge is the slow part; a long silent interval during an SVD is expected.

In [ ]:
result_s2025=run_and_download_stream('s2025',2025)

## Stream s2026
The previous stream is already an atomic JSON. If the runtime resets, restart from cell 1 and upload that JSON together with the M23 ZIP.

In [ ]:
result_s2026=run_and_download_stream('s2026',2026)

## Stream s2027

In [ ]:
result_s2027=run_and_download_stream('s2027',2027)

In [ ]:
# Verify all handoffs, aggregate mean/sample-std, and export evidence only.
command=[sys.executable,'-u',RUNNER,'summarize','--config',CONFIG,'--source-m23-artifact',str(M23_PATH),'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'m24_results.json'
assert result_path.is_file(),'M24 failed before writing the aggregate.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status']); print('AGGREGATE:',json.dumps(result['aggregate'],indent=2)); print('PAIRED:',json.dumps(result['paired_comparisons'],indent=2)); print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='PASS_M24_EQUAL_MEMORY_CONTROLS_TRAIN_ONLY','M24 aggregate failed structural gates.'
bundle=Path('/content/srq_generalization_m24_equal_memory_controls_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(result_path,bundle/'m24_results.json')
for path in sorted((Path(OUTPUT_DIR)/'m24_results').glob('stream_*_results.json')): shutil.copy2(path,bundle/path.name)
shutil.copy2(Path(CONFIG),bundle/'config.json')
archive=shutil.make_archive(FINAL_EXPORT[:-4],'zip',root_dir=bundle)
print('ARTIFACT:',archive,'SHA256:',sha_raw(archive),'SIZE:',Path(archive).stat().st_size)
files.download(archive)